Here I will take the stocks chosen from the reduced daily dataset, and obtain hourly data.

In [11]:
import pandas as pd

In [13]:
df_reduced_daily_stocks = pd.read_csv('transformed_daily_data/dailydata_5years_30ktransacs.csv')
df_reduced_daily_stocks

,timestamp,ticker,open,high,low,close,volume,vwap,transactions
0,2021-05-05,AA,39.37,40.880,38.640,40.60,7.773673e+06,40.1427,54715.0
1,2021-05-06,AA,40.89,41.440,39.280,39.89,7.308031e+06,40.0879,52580.0
2,2021-05-07,AA,39.76,41.380,39.250,41.20,4.909460e+06,40.6481,40057.0
3,2021-05-10,AA,42.75,44.420,41.020,41.04,1.183422e+07,42.5571,81210.0
4,2021-05-11,AA,39.07,42.150,38.960,41.32,9.246082e+06,40.8798,71792.0
...,...,...,...,...,...,...,...,...,...
497833,2026-04-27,ZTS,116.62,119.680,116.600,117.87,3.160355e+06,118.2275,43149.0
497834,2026-04-28,ZTS,117.23,118.290,116.075,116.65,2.972547e+06,117.0538,47388.0
497835,2026-04-29,ZTS,115.50,115.905,112.970,113.83,3.401808e+06,113.9745,47839.0
497836,2026-04-30,ZTS,114.04,115.535,113.790,114.97,4.297206e+06,114.7143,57409.0


In [5]:
stocks_to_fetch = df_reduced_daily_stocks['ticker'].unique().tolist()


In [6]:
stocks_to_fetch

['AA',
 'AAL',
 'AAPL',
 'ABBV',
 'ABNB',
 'ABT',
 'ACN',
 'ADBE',
 'ADI',
 'ADM',
 'ADP',
 'ADSK',
 'AEM',
 'AEO',
 'AEP',
 'AES',
 'AFL',
 'AFRM',
 'AG',
 'AGNC',
 'AI',
 'AIG',
 'ALB',
 'ALLY',
 'AMAT',
 'AMC',
 'AMCR',
 'AMD',
 'AMGN',
 'AMT',
 'AMZN',
 'ANET',
 'APA',
 'APH',
 'APO',
 'APP',
 'APTV',
 'AR',
 'ARRY',
 'ASAN',
 'ASTS',
 'AVGO',
 'AVTR',
 'AXP',
 'AZN',
 'BA',
 'BAC',
 'BAX',
 'BBY',
 'BE',
 'BEN',
 'BILL',
 'BK',
 'BKR',
 'BLDR',
 'BMY',
 'BRK.B',
 'BSX',
 'BTU',
 'BX',
 'BYND',
 'C',
 'CAG',
 'CAH',
 'CARR',
 'CAT',
 'CB',
 'CCI',
 'CCJ',
 'CCL',
 'CDE',
 'CDNS',
 'CELH',
 'CF',
 'CFG',
 'CHPT',
 'CHTR',
 'CHWY',
 'CI',
 'CL',
 'CLF',
 'CLOV',
 'CMCSA',
 'CME',
 'CMG',
 'CNC',
 'CNP',
 'CNQ',
 'COF',
 'COIN',
 'COP',
 'COST',
 'CPB',
 'CPNG',
 'CPRT',
 'CRH',
 'CRM',
 'CRWD',
 'CSCO',
 'CSGP',
 'CSX',
 'CTSH',
 'CTVA',
 'CVE',
 'CVNA',
 'CVS',
 'CVX',
 'CZR',
 'D',
 'DAL',
 'DASH',
 'DBX',
 'DD',
 'DDOG',
 'DE',
 'DELL',
 'DG',
 'DHI',
 'DHR',
 'DIS',
 'DKNG',
 'DL

In [8]:
df_reduced_daily_stocks['ticker'].nunique()

397

In my smaller version, I set 5000 hours. With 7 trading hours each day that would equate to about 714 days. 

With 30 stocks, that gave me 150000 rows, and that used I believe under 20% of my API credits for the day. (750000 rows with the same columns is the threshold I will plan for).

Now with 397 stocks, I can comfortably get 1889 hours of data. At 7 trading hours a day that's 269 days. 

I'll go with 180 days, which is about 1260 hours, I'll round up to 1270 to ensure no mid-day cutoffs.

In [9]:
import requests
import pandas as pd
from pathlib import Path
import time

def load_api_key(filepath="api_keys/twelvedata.txt"):
    return Path(filepath).read_text(encoding="utf-8").strip()


def get_twelve_data_hourly(ticker, api_key, outputsize=5000):
    url = "https://api.twelvedata.com/time_series"

    params = {
        "symbol": ticker,
        "interval": "1h",
        "outputsize": outputsize,
        "apikey": api_key,
    }

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()

    if "values" not in data:
        raise ValueError(f"Twelve Data error for {ticker}: {data}")

    df = pd.DataFrame(data["values"])

    df = df.rename(columns={
        "datetime": "timestamp",
        "open": "open",
        "high": "high",
        "low": "low",
        "close": "close",
        "volume": "volume",
    })

    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["ticker"] = ticker

    for col in ["open", "high", "low", "close", "volume"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["vwap"] = pd.NA
    df["transactions"] = pd.NA

    df = df[
        ["timestamp", "ticker", "open", "high", "low", "close", "volume", "vwap", "transactions"]
    ]

    return df.sort_values("timestamp").reset_index(drop=True)


def download_twelve_data_hourly_for_tickers(
    tickers,
    api_key_path="api_keys/twelve_data.txt",
    save_path="raw_hourly_data.csv",
    sleep_seconds=8
):
    api_key = load_api_key(api_key_path)
    all_dfs = []

    for ticker in tickers:
        print(f"Downloading {ticker}...")
        df = get_twelve_data_hourly(ticker, api_key)
        all_dfs.append(df)
        time.sleep(sleep_seconds)

    combined_df = pd.concat(all_dfs, ignore_index=True)
    combined_df = combined_df.sort_values(["ticker", "timestamp"]).reset_index(drop=True)

    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    combined_df.to_csv(save_path, index=False)

    print(f"Saved {len(combined_df):,} rows to {save_path}")

    return combined_df

In [10]:
# I'll call this list directly from the other dataframe
tickers = df_reduced_daily_stocks['ticker'].unique().tolist()



hourly_df = download_twelve_data_hourly_for_tickers(
    tickers=tickers,
    api_key_path="api_keys/twelvedata.txt",
    save_path="raw_hourly_data.csv",
    sleep_seconds=8
)

hourly_df.head()

Saved 1,985,000 rows to raw_hourly_data.csv


,timestamp,ticker,open,high,low,close,volume,vwap,transactions
0,2023-06-23 15:30:00,AA,32.590000,32.89,32.580000,32.820000,1056860,NaN,NaN
1,2023-06-26 09:30:00,AA,32.780000,33.53,32.780000,33.380000,563671,NaN,NaN
2,2023-06-26 10:30:00,AA,33.390000,33.39,32.990000,33.043499,396548,NaN,NaN
3,2023-06-26 11:30:00,AA,33.060001,33.46,33.000000,33.242900,459899,NaN,NaN
4,2023-06-26 12:30:00,AA,33.259900,33.30,33.099998,33.190000,293175,NaN,NaN


In [14]:
import pandas as pd
df = pd.read_csv('raw_hourly_data/raw_hourly_data_397_stocks.csv')
df

,timestamp,ticker,open,high,low,close,volume,vwap,transactions
0,2023-06-23 15:30:00,AA,32.590000,32.89,32.580000,32.820000,1056860,NaN,NaN
1,2023-06-26 09:30:00,AA,32.780000,33.53,32.780000,33.380000,563671,NaN,NaN
2,2023-06-26 10:30:00,AA,33.390000,33.39,32.990000,33.043499,396548,NaN,NaN
3,2023-06-26 11:30:00,AA,33.060001,33.46,33.000000,33.242900,459899,NaN,NaN
4,2023-06-26 12:30:00,AA,33.259900,33.30,33.099998,33.190000,293175,NaN,NaN
...,...,...,...,...,...,...,...,...,...
1984995,2026-05-04 11:30:00,ZTS,113.040001,113.12,112.185000,112.185000,285011,NaN,NaN
1984996,2026-05-04 12:30:00,ZTS,112.180000,113.19,112.099998,113.080002,259883,NaN,NaN
1984997,2026-05-04 13:30:00,ZTS,113.059998,113.17,112.740000,113.004997,243781,NaN,NaN
1984998,2026-05-04 14:30:00,ZTS,112.985000,113.20,112.320000,112.350000,371968,NaN,NaN


In [15]:
df['timestamp'].nunique()

6923

In [16]:
df['ticker'].nunique()

397

In [17]:
df['ticker'].value_counts()

ticker
AA      5000
OPEN    5000
ON      5000
OMC     5000
OKTA    5000
        ... 
EL      5000
EBAY    5000
EA      5000
DXCM    5000
ZTS     5000
Name: count, Length: 397, dtype: int64

397 stocks with 5000 hours gave me about 2 million rows and used about half of my free API credits for the day.

If I went with 6000 stocks, I'd have to reduce to 333 trading hours (47 trading days) to achieve the same data size. Could maybe double that to 630 trading hours to give me 90 trading days and just get under the wire for API credits.